In [ ]:
#provied utility code
import torch
import torch.nn as nn
import torchvision
import copy

# ------------------------------------------------------------------
# 1. Fuse a single Conv2d + BatchNorm2d pair
#    (Same fuse_conv_and_bn function as before)
# ------------------------------------------------------------------
def fuse_conv_and_bn(conv, bn):
    """
    Takes a conv layer and a batch norm layer, and returns
    a new conv layer with BN 'baked in'.
    """
    # Initialize fused Conv2d (must have bias because BN modifies bias)
    fusedconv = nn.Conv2d(
        in_channels=conv.in_channels,
        out_channels=conv.out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        groups=conv.groups,
        bias=True
    )

    # Clone conv weight
    w_conv = conv.weight.clone().view(conv.out_channels, -1)
    # BN scale factor for each channel: gamma / sqrt(var + eps)
    w_bn = torch.diag(bn.weight.div(torch.sqrt(bn.running_var + bn.eps)))
    # Fused weights
    fused_weight = torch.mm(w_bn, w_conv).view(conv.weight.size())
    fusedconv.weight = nn.Parameter(fused_weight)

    # Construct fused bias
    if conv.bias is not None:
        b_conv = conv.bias
    else:
        b_conv = torch.zeros(conv.out_channels, dtype=conv.weight.dtype, device=conv.weight.device)

    b_bn = bn.bias - bn.weight.mul(bn.running_mean).div(torch.sqrt(bn.running_var + bn.eps))
    fusedconv.bias = nn.Parameter(b_conv + b_bn)

    return fusedconv


# ------------------------------------------------------------------
# 2. Recursively fuse all Conv+BN pairs in a ResNet18 model
#    This function will modify the model in-place.
# ------------------------------------------------------------------
def fuse_resnet_conv_bn_inplace(module):
    """
    Recursively fuses all Conv2d+BatchNorm2d pairs in a (sub)module
    of a ResNet. Typical use: fuse_resnet_conv_bn_inplace(resnet18_model).
    """
    # If the module has children (like BasicBlock, etc.), we recurse down
    for name, child in list(module.named_children()):
        # 1) Recurse first so you go down into submodules (e.g., BasicBlock)
        fuse_resnet_conv_bn_inplace(child)

        # 2) Then check if child is exactly a BasicBlock or Bottleneck
        #    Because typical ResNet blocks have conv1+bn1, conv2+bn2, ...
        if isinstance(child, nn.Sequential):
            # E.g. the downsample in BasicBlock is often a Sequential of {Conv2d, BN2d}
            # but let's fuse inside that if it's just a two-layer sequence
            # We'll handle downsample as well
            if len(child) == 2:
                if isinstance(child[0], nn.Conv2d) and isinstance(child[1], nn.BatchNorm2d):
                    fused = fuse_conv_and_bn(child[0], child[1])
                    setattr(module, name, fused)  # replace child with fused conv
        elif isinstance(child, nn.BatchNorm2d):
            # If you see a BN without an immediate preceding conv in the same submodule,
            # it's typically because the ResNet BasicBlock has separate attributes (conv1,bn1), (conv2,bn2).
            # We'll try to fuse with the conv that appears just before it in the parent.
            # That logic requires we also look at the parent's attributes in order, or the block's structure.

            # Example: In BasicBlock:
            #   self.conv1 = nn.Conv2d(...)
            #   self.bn1   = nn.BatchNorm2d(...)
            #   self.conv2 = nn.Conv2d(...)
            #   self.bn2   = nn.BatchNorm2d(...)
            # So if "child" is "bn1", the sibling "conv1" might exist in the same module's __dict__.

            # Because we just have the child, let's see if there's a "conv" with the same suffix in the name:
            #   e.g. bn1 -> conv1, bn2 -> conv2
            # This is a bit hacky, but it's how the basic blocks are typically named.
            bn_name = name
            suffix = bn_name[-1]   # '1' or '2'
            conv_name = "conv" + suffix

            # If the sibling conv exists, fuse them
            if hasattr(module, conv_name):
                conv_sibling = getattr(module, conv_name)
                if isinstance(conv_sibling, nn.Conv2d):
                    # fuse them
                    fused = fuse_conv_and_bn(conv_sibling, child)
                    # replace the conv
                    setattr(module, conv_name, fused)
                    # The BN is now baked-in, so let's set BN to Identity
                    identity = nn.Identity()
                    setattr(module, bn_name, identity)

    return module


# ------------------------------------------------------------------
# 3. Main code to demonstrate fusing ResNet18
# ------------------------------------------------------------------
if __name__ == "__main__":
    # (A) Create a ResNet18 model from torchvision
    original_model = torchvision.models.resnet18(pretrained=False)
    original_model.eval()

    # Make a copy so we can compare original vs. fused
    fused_model = copy.deepcopy(original_model)

    # (B) Fuse in place
    fuse_resnet_conv_bn_inplace(fused_model)
    fused_model.eval()

    # (C) Validate: same output on random input
    # Create random input
    torch.manual_seed(0)
    x = torch.randn(2, 3, 224, 224)


    with torch.no_grad():
        out_original = original_model(x)
        out_fused = fused_model(x)

    # Compare
    diff = (out_original - out_fused).abs().max().item()
    print(f"Max difference between original ResNet18 and fused ResNet18: {diff:.6g}")
    # Expect the difference to be on the order of 1e-6 ~ 1e-7
    # depending on floating point precision.

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Max difference between original ResNet18 and fused ResNet18: 2.14577e-06


In [ ]:
#provided utility function to convert numpy to bin and vice versa
import numpy as np

def npy_to_bin(input_npy_path, output_bin_path, dtype='float32'):
    arr = np.load(input_npy_path).astype(dtype)
    arr.tofile(output_bin_path)
    print(f"Saved binary to: {output_bin_path}")

def bin_to_npy(input_bin_path, output_npy_path, shape=None, dtype='float32'):
    data = np.fromfile(input_bin_path, dtype=dtype)
    if shape is not None:
        data = data.reshape(shape)
    np.save(output_npy_path, data)
    print(f"Saved .npy to: {output_npy_path}, with shape: {data.shape}")

In [ ]:
bin_to_npy('/content/drive/MyDrive/8893/layer4_block0_downsample_weights.bin', '/content/drive/MyDrive/8893/layer4_block0_downsample_weights.npy', dtype='float32',shape=(512,256,1,1))
bin_to_npy('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_beta.bin', '/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_beta.npy', dtype='float32')
bin_to_npy('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_gamma.bin', '/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_gamma.npy', dtype='float32')
bin_to_npy('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_mean.bin', '/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_mean.npy', dtype='float32')
bin_to_npy('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_var.bin', '/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_var.npy', dtype='float32')
#npy_to_bin('/content/drive/MyDrive/8893/results/layer4_block0_relu1_output.npy', '/content/drive/MyDrive/8893/results/layer4_block0_relu1_output.bin', dtype='float32')

Saved .npy to: /content/drive/MyDrive/8893/layer4_block0_downsample_weights.npy, with shape: (512, 256, 1, 1)
Saved .npy to: /content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_beta.npy, with shape: (512,)
Saved .npy to: /content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_gamma.npy, with shape: (512,)
Saved .npy to: /content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_mean.npy, with shape: (512,)
Saved .npy to: /content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_var.npy, with shape: (512,)


In [ ]:
#custom code to retrieve fused data from existing bin files
import numpy as np
import torch

def fuse_conv_bn_numpy(conv_weight, conv_bias, bn_gamma, bn_beta, bn_mean, bn_var, eps=1e-5):
    """
    Fuse Conv2d + BatchNorm2d weights without needing the full model.

    Args:
        conv_weight: numpy array of shape [out_c, in_c, kH, kW]
        conv_bias: numpy array of shape [out_c] (or None)
        bn_gamma: numpy array of shape [out_c] (BN weights)
        bn_beta: numpy array of shape [out_c] (BN bias)
        bn_mean: numpy array of shape [out_c] (BN running mean)
        bn_var: numpy array of shape [out_c] (BN running variance)
        eps: BN epsilon

    Returns:
        fused_weight, fused_bias as numpy arrays
    """
    # Convert numpy arrays to torch tensors
    conv_weight = torch.from_numpy(conv_weight)
    bn_gamma = torch.from_numpy(bn_gamma)
    bn_beta = torch.from_numpy(bn_beta)
    bn_mean = torch.from_numpy(bn_mean)
    bn_var = torch.from_numpy(bn_var)

    if conv_bias is not None:
        conv_bias = torch.from_numpy(conv_bias)
    else:
        conv_bias = torch.zeros_like(bn_mean)

    # Reshape conv weights to [out_c, in_c*kH*kW]
    out_c = conv_weight.shape[0]
    w_conv = conv_weight.reshape(out_c, -1)

    # Compute BN scaling factor: gamma / sqrt(var + eps)
    scale_factor = bn_gamma / torch.sqrt(bn_var + eps)

    # Fuse weights: diag(scale_factor) @ w_conv
    fused_weight = torch.diag(scale_factor) @ w_conv
    fused_weight = fused_weight.reshape(conv_weight.shape)

    # Fuse biases: (conv_bias - bn_mean) * scale_factor + bn_beta
    fused_bias = (conv_bias - bn_mean) * scale_factor + bn_beta

    # Convert back to numpy
    return fused_weight.numpy(), fused_bias.numpy()

In [ ]:
# Load your numpy arrays
conv_weight = np.load('/content/drive/MyDrive/8893/layer4_block0_downsample_weights.npy')  # [out_c, in_c, kH, kW]
conv_bias = None       # [out_c] (or None)
bn_gamma = np.load('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_gamma.npy')         # [out_c]
bn_beta = np.load('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_beta.npy')           # [out_c]
bn_mean = np.load('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_mean.npy')           # [out_c]
bn_var = np.load('/content/drive/MyDrive/8893/layer4_block0_downsample_batch_norm_var.npy')             # [out_c]

# Fuse parameters
fused_weight, fused_bias = fuse_conv_bn_numpy(
    conv_weight=conv_weight,
    conv_bias=conv_bias,  # can be None
    bn_gamma=bn_gamma,
    bn_beta=bn_beta,
    bn_mean=bn_mean,
    bn_var=bn_var
)

# Save fused weights
np.save('block0_downsample_fused_conv_weight.npy', fused_weight)
np.save('block0_downsample_fused_conv_bias.npy', fused_bias)

In [ ]:
fused_weight.shape

(512, 512, 3, 3)

In [ ]:
fused_bias.shape

(512,)

In [ ]:
npy_to_bin('block1_fused_conv1_weight.npy','/content/drive/MyDrive/8893/fused bin/block1_fused_conv1_weight.bin', dtype='float32')
npy_to_bin('block1_fused_conv1_bias.npy','/content/drive/MyDrive/8893/fused bin/block1_fused_conv1_bias.bin', dtype='float32')

Saved binary to: /content/drive/MyDrive/8893/fused bin/block1_fused_conv1_weight.bin
Saved binary to: /content/drive/MyDrive/8893/fused bin/block1_fused_conv1_bias.bin
